# Lab 4.2: Structured Error Responses for MCP Tools

**What you'll build:** A structured error response system for MCP tools that enables intelligent agent recovery -- and learn why plain error strings cause agents to make bad decisions.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- plain string errors cause agent confusion | 5 min |
| 2 | The Right Way -- structured errors enable intelligent recovery | 5 min |
| 3 | Your Turn -- build structured error responses for a file management MCP tool | 10 min |
| 4 | Stress Test -- verify the agent makes correct recovery decisions | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building MCP tools for a file management system. The tools can fail in several ways: file not found, permission denied, rate limited by a cloud API, or internal server errors. The agent calling these tools needs to know:

1. **Did it fail?** (isError)
2. **What kind of failure?** (errorCategory)
3. **Should I retry?** (isRetryable + retryAfterMs)
4. **What do I tell the user?** (userMessage)

We'll simulate tool failures and see how Claude responds to unstructured vs structured errors.

In [ ]:
# Simulated error scenarios
ERROR_SCENARIOS = [
    {
        "name": "file_not_found",
        "description": "User asks to read a file that doesn't exist",
        "should_retry": False,
        "correct_action": "Tell user the file doesn't exist, ask for correct path",
    },
    {
        "name": "rate_limited",
        "description": "Cloud storage API returns 429 Too Many Requests",
        "should_retry": True,
        "correct_action": "Wait and retry after the specified delay",
    },
    {
        "name": "permission_denied",
        "description": "User lacks access to the requested file",
        "should_retry": False,
        "correct_action": "Tell user they don't have permission, suggest requesting access",
    },
    {
        "name": "timeout",
        "description": "Network timeout connecting to storage backend",
        "should_retry": True,
        "correct_action": "Retry the request (network timeouts are transient)",
    },
]

print("Error scenarios loaded:")
for s in ERROR_SCENARIOS:
    retry_tag = "RETRY" if s["should_retry"] else "DO NOT RETRY"
    print(f"  {s['name']}: {retry_tag} -- {s['correct_action']}")

---

## Phase 1: The Wrong Approach

Most MCP tools return plain error strings. Let's see how Claude handles them.

In [ ]:
# Simulate unstructured error responses
UNSTRUCTURED_ERRORS = {
    "file_not_found": "Error: File '/data/report.csv' not found.",
    "rate_limited": "Error: Too many requests. Please try again later.",
    "permission_denied": "Error: Permission denied for '/secret/config.yaml'.",
    "timeout": "Error: Connection timed out after 30 seconds.",
}

# Ask Claude what it would do with each unstructured error
TOOLS = [{
    "name": "read_file",
    "description": "Reads a file from the file system.",
    "input_schema": {
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "File path to read"}
        },
        "required": ["path"]
    }
}]

print("Testing Claude's recovery decisions with UNSTRUCTURED errors:\n")

for scenario in ERROR_SCENARIOS:
    error_text = UNSTRUCTURED_ERRORS[scenario["name"]]
    
    # Simulate: user asked to read a file, tool returned an error
    messages = [
        {"role": "user", "content": "Please read the file /data/report.csv"},
        {"role": "assistant", "content": [{"type": "tool_use", "id": "call_1", "name": "read_file", "input": {"path": "/data/report.csv"}}]},
        {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "call_1", "content": error_text, "is_error": True}]},
    ]
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        tools=TOOLS,
        messages=messages,
        system="You are an assistant. When a tool fails, decide: (1) retry the same call, (2) try a different approach, or (3) tell the user. State your decision and reasoning in one sentence."
    )
    
    decision = response.content[0].text if response.content[0].type == "text" else "[tool call]"
    would_retry = response.stop_reason == "tool_use"
    
    correct = would_retry == scenario["should_retry"]
    tag = "CORRECT" if correct else "WRONG"
    
    print(f"  [{tag}] {scenario['name']}")
    print(f"    Error: {error_text}")
    print(f"    Claude decided: {'RETRY' if would_retry else 'ESCALATE'}")
    print(f"    Should: {'RETRY' if scenario['should_retry'] else 'ESCALATE'}")
    print(f"    Response: {decision[:100]}")
    print()

### Why unstructured errors fail

- **No retry signal.** Claude has to guess from the error string whether retrying will help. "Connection timed out" and "file not found" look equally bad as strings, but one is transient and one is permanent.
- **No category routing.** The agent cannot apply different recovery strategies per error type -- it treats all errors the same.
- **No backoff hint.** Even when Claude correctly decides to retry, it has no idea how long to wait.
- **No user-safe message.** The raw error string may contain internal details that should not be shown to end users.

---

## Phase 2: The Right Approach

Structured error responses give the agent machine-parseable metadata to make correct recovery decisions.

In [ ]:
# Structured error responses
STRUCTURED_ERRORS = {
    "file_not_found": json.dumps({
        "errorCategory": "not_found",
        "isRetryable": False,
        "userMessage": "The file '/data/report.csv' was not found. Please check the file path and try again.",
        "details": {"path": "/data/report.csv", "suggestion": "Check if the file exists or verify the path"}
    }),
    "rate_limited": json.dumps({
        "errorCategory": "rate_limit",
        "isRetryable": True,
        "retryAfterMs": 5000,
        "userMessage": "The service is temporarily busy. Retrying in a few seconds.",
        "details": {"limit": "100 requests/minute", "reset_at": "2024-01-15T10:30:00Z"}
    }),
    "permission_denied": json.dumps({
        "errorCategory": "authorization",
        "isRetryable": False,
        "userMessage": "You don't have permission to access '/secret/config.yaml'. Please request access from your administrator.",
        "details": {"path": "/secret/config.yaml", "required_role": "admin"}
    }),
    "timeout": json.dumps({
        "errorCategory": "timeout",
        "isRetryable": True,
        "retryAfterMs": 2000,
        "userMessage": "The connection timed out. Retrying automatically.",
        "details": {"timeout_ms": 30000, "endpoint": "storage-backend"}
    }),
}

print("Testing Claude's recovery decisions with STRUCTURED errors:\n")

for scenario in ERROR_SCENARIOS:
    error_text = STRUCTURED_ERRORS[scenario["name"]]
    
    messages = [
        {"role": "user", "content": "Please read the file /data/report.csv"},
        {"role": "assistant", "content": [{"type": "tool_use", "id": "call_1", "name": "read_file", "input": {"path": "/data/report.csv"}}]},
        {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "call_1", "content": error_text, "is_error": True}]},
    ]
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        tools=TOOLS,
        messages=messages,
        system="You are an assistant. When a tool fails, examine the structured error fields (errorCategory, isRetryable, retryAfterMs). If isRetryable is true, retry. If false, inform the user using the userMessage. State your decision in one sentence."
    )
    
    decision = response.content[0].text if response.content[0].type == "text" else "[tool call]"
    would_retry = response.stop_reason == "tool_use"
    
    correct = would_retry == scenario["should_retry"]
    tag = "CORRECT" if correct else "WRONG"
    
    print(f"  [{tag}] {scenario['name']}")
    print(f"    Claude decided: {'RETRY' if would_retry else 'ESCALATE'}")
    print(f"    Should: {'RETRY' if scenario['should_retry'] else 'ESCALATE'}")
    print(f"    Response: {decision[:100]}")
    print()

---

## Phase 3: Your Turn

Build a structured error response function for a database MCP tool. Each error type needs the correct metadata.

**Your goal:** Every error returns the correct isRetryable value and a useful errorCategory.

In [ ]:
CHALLENGE_SCENARIOS = [
    {"error_type": "connection_refused", "http_status": None, "should_retry": True, "category": "connection"},
    {"error_type": "invalid_query", "http_status": 400, "should_retry": False, "category": "validation"},
    {"error_type": "table_not_found", "http_status": 404, "should_retry": False, "category": "not_found"},
    {"error_type": "auth_token_expired", "http_status": 401, "should_retry": False, "category": "authentication"},
    {"error_type": "server_overloaded", "http_status": 503, "should_retry": True, "category": "rate_limit"},
    {"error_type": "deadlock_detected", "http_status": 500, "should_retry": True, "category": "internal"},
]

print("Challenge: Build structured error responses for a database MCP tool.")
print(f"Error types to handle: {[s['error_type'] for s in CHALLENGE_SCENARIOS]}")

In [ ]:
# =============================================================
# TODO: Build a structured error response function
# =============================================================
#
# Requirements:
# - Return a dict with: errorCategory, isRetryable, userMessage
# - For retryable errors, include retryAfterMs
# - userMessage should be safe to show end users (no internal details)
#
# Think about:
# - Which errors are transient (retry) vs permanent (escalate)?
# - What backoff time makes sense for each retryable error?
# - What user message helps without exposing internals?

def build_error_response(error_type: str, details: dict = None) -> dict:
    """
    Build a structured MCP error response.
    
    Args:
        error_type: One of the error types from CHALLENGE_SCENARIOS
        details: Optional additional error details
    
    Returns:
        A dict with errorCategory, isRetryable, userMessage, and optionally retryAfterMs
    """
    # TODO: Implement this function
    # Hint: Use a mapping from error_type to structured response fields
    raise NotImplementedError("Implement build_error_response()")


# Test your function
print("Testing your error responses:\n")
results = []
for scenario in CHALLENGE_SCENARIOS:
    try:
        response = build_error_response(scenario["error_type"])
        retry_correct = response.get("isRetryable") == scenario["should_retry"]
        cat_correct = response.get("errorCategory") == scenario["category"]
        has_message = bool(response.get("userMessage"))
        has_backoff = not scenario["should_retry"] or "retryAfterMs" in response
        
        all_correct = retry_correct and cat_correct and has_message and has_backoff
        tag = "PASS" if all_correct else "FAIL"
        results.append(all_correct)
        
        print(f"  [{tag}] {scenario['error_type']}")
        if not retry_correct:
            print(f"    isRetryable: got {response.get('isRetryable')}, expected {scenario['should_retry']}")
        if not cat_correct:
            print(f"    errorCategory: got {response.get('errorCategory')}, expected {scenario['category']}")
        if not has_message:
            print(f"    Missing userMessage")
        if not has_backoff:
            print(f"    Missing retryAfterMs for retryable error")
    except NotImplementedError:
        print(f"  [SKIP] {scenario['error_type']} -- not implemented yet")
        results.append(False)

passed = sum(results)
total = len(results)
print(f"\nScore: {passed}/{total}")
if passed == total:
    print("PASSED -- all error responses correctly structured!")

---

## Phase 4: Stress Test

Feed your structured errors to Claude and verify it makes the correct retry/escalate decisions.

In [ ]:
DB_TOOL = {
    "name": "query_database",
    "description": "Executes a SQL query against the application database.",
    "input_schema": {
        "type": "object",
        "properties": {
            "sql": {"type": "string", "description": "SQL query to execute"},
            "database": {"type": "string", "description": "Database name"}
        },
        "required": ["sql"]
    }
}

print("Stress test: Does Claude make correct decisions with your structured errors?\n")

correct_decisions = 0
total_tests = 0

for scenario in CHALLENGE_SCENARIOS:
    try:
        error_resp = build_error_response(scenario["error_type"])
    except NotImplementedError:
        continue
    
    error_json = json.dumps(error_resp)
    
    messages = [
        {"role": "user", "content": "Run SELECT count(*) FROM users"},
        {"role": "assistant", "content": [{"type": "tool_use", "id": "call_1", "name": "query_database", "input": {"sql": "SELECT count(*) FROM users"}}]},
        {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "call_1", "content": error_json, "is_error": True}]},
    ]
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        tools=[DB_TOOL],
        messages=messages,
        system="When a tool fails, check isRetryable. If true, retry the tool. If false, inform the user with the userMessage."
    )
    
    would_retry = response.stop_reason == "tool_use"
    correct = would_retry == scenario["should_retry"]
    correct_decisions += int(correct)
    total_tests += 1
    
    tag = "CORRECT" if correct else "WRONG"
    print(f"  [{tag}] {scenario['error_type']}: Claude {'retried' if would_retry else 'escalated'} (should {'retry' if scenario['should_retry'] else 'escalate'})")

if total_tests > 0:
    print(f"\nDecision accuracy: {correct_decisions}/{total_tests} ({correct_decisions/total_tests:.0%})")
    if correct_decisions == total_tests:
        print("Your structured errors enable perfect agent recovery decisions!")

### Key Takeaways

1. **Plain error strings force the agent to guess.** Without structured metadata, the agent cannot reliably distinguish transient from permanent failures.
2. **isRetryable is the critical field.** It is the binary signal the agent needs to decide retry vs escalate. Without it, agents either retry everything or nothing.
3. **errorCategory enables category-specific strategies.** Rate limits need backoff, auth failures need re-authentication, not-found needs path correction.
4. **userMessage separates internal details from user-facing communication.** Log stack traces server-side; return human-safe messages to the agent.
5. **Local recovery first, propagation second.** The agent should try to recover (retry, fallback) before escalating to the user.